# PoC step 3: reconcile, post to the ledger, score
Matches the bank and wallet legs, posts the settlement journal, fills the exception queue, updates statuses, and scores everything against the manifest.

**Before you start:** copy `recon.py` and `test_recon.py` into `bw-interop-poc/src`. Step 2 must have created `data/poc.db`.

In [1]:
# Cell 1: mount Drive, set paths, install test dependency
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/bw-interop-poc'
!pip -q install pytest

Mounted at /content/drive


In [2]:
# Cell 2: run all tests (expect 36 + 14 = 50 passed)
!cd {BASE}/src && python -m pytest -q -p no:cacheprovider

..................................................                       [100%]
50 passed in 6.77s


In [3]:
# Cell 3: reconcile (safe to re-run: --rerun clears the previous reconciliation first)
!python {BASE}/src/recon.py --rerun --db {BASE}/data/poc.db --manifest {BASE}/data/poc_manifest.csv --out {BASE}/data/recon

Reconciliation outcomes: {'MATCHED': 352, 'DISCREPANT': 24, 'UNMATCHED': 24}
Journal entries: 704 | unbalanced transactions: 0
Accuracy vs manifest: 408/408 = 100.00% (false matches 0, missed matches 0)
                  correct    n
(clean)               336  336
AMOUNT_MISMATCH        12   12
DUPLICATE_E2E          16   16
FAILED_STATUS          12   12
LAG_OUTLIER             8    8
MISSING_CURRENCY        6    6
ORPHAN_ISO              6    6
ORPHAN_OM               6    6
UNKNOWN_BIC             6    6


In [4]:
# Cell 4: check and look at the results
import json, sqlite3
import pandas as pd
summary = json.load(open(f'{BASE}/data/recon_summary.json'))
print('Accuracy vs manifest:', summary['correct'], '/', summary['rows'], '=', summary['accuracy'])
assert (summary['rows'], summary['correct']) == (408, 408), 'Some rows differ from the manifest: see recon_results.csv'
print(pd.DataFrame(summary['confusion']).fillna(0).astype(int).T.to_string())   # rows = expected, columns = actual
db = sqlite3.connect(f'{BASE}/data/poc.db')
print(pd.read_sql("SELECT outcome, match_method, COUNT(*) AS n FROM reconciliation_records GROUP BY outcome, match_method", db).to_string(index=False))
print(pd.read_sql("SELECT reason_code, COUNT(*) AS n FROM exceptions WHERE source_system='RECON' GROUP BY reason_code", db).to_string(index=False))
print(pd.read_sql("SELECT status, status_source, COUNT(*) AS n FROM transactions WHERE source_system='PACS008' GROUP BY status, status_source", db).to_string(index=False))
pd.read_csv(f'{BASE}/data/recon_results.csv').head(8)

Accuracy vs manifest: 408 / 408 = 1.0
            DISCREPANT  MATCHED  REJECTED  UNMATCHED
DISCREPANT          24        0         0          0
MATCHED              0      352         0          0
REJECTED             0        0        20          0
UNMATCHED            0        0         0         12
   outcome match_method   n
DISCREPANT          KEY  24
   MATCHED          KEY 352
 UNMATCHED         NONE  24
    reason_code  n
    AMOUNT_DIFF 12
    LAG_OUTLIER  8
    NO_BANK_LEG 18
  NO_WALLET_LEG  6
STATUS_MISMATCH 12
    status  status_source   n
 COMPLETED DERIVED_LEDGER 352
DISPATCHED DERIVED_LEDGER  18
    FAILED DERIVED_LEDGER  12


,pair_id,end_to_end_id,anomaly_type,expected_outcome,actual_outcome,expected_flag,actual_flags,outcome_correct,flag_correct,lag_matches_manifest,correct,amount_diff_minor,day_gap,match_method
0,P0001,E2E-BW-20260807-0001,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,1.0,KEY
1,P0002,E2E-BW-20260830-0002,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,3.0,KEY
2,P0003,E2E-BW-20260810-0003,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,0.0,KEY
3,P0004,E2E-BW-20260816-0004,MISSING_CURRENCY,REJECTED,REJECTED,NaN,MISSING_CURRENCY,True,True,True,True,NaN,NaN,NaN
4,P0005,E2E-BW-20260809-0005,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,2.0,KEY
5,P0006,E2E-BW-20260811-0006,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,3.0,KEY
6,P0007,E2E-BW-20260813-0007,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,3.0,KEY
7,P0008,E2E-BW-20260806-0008,(clean),MATCHED,MATCHED,NaN,NaN,True,True,True,True,0.0,0.0,KEY


## Done when
- Cell 2 ends with `50 passed`.
- Cell 3 prints `MATCHED: 352, DISCREPANT: 24, UNMATCHED: 24`, `Journal entries: 704 | unbalanced transactions: 0`, and `408/408`.
- Cell 4 runs without an error.

The file `data/recon_results.csv` is the input for the Power BI PoC page. Paste the printed output of cells 2, 3, and 4 into the decision log.

**Read this before quoting the 100%:** the manifest and the engine are built from the same rules, so 408/408 shows that the implementation follows the specification. It is not a measure of accuracy on real payments. State this in the Methodology and Discussion.